# TalentoBR - CV Screener v0.4 (Data Team, início deste ano)

Protótipo de triagem de currículos contra descrição de vaga.

**Status:** experimental. Não usar em produção.  
**Autor:** Data Team — TalentoBR  
**Aviso:** dados utilizados em `data/cvs-exemplos/` são SINTÉTICOS.

In [ ]:
import os
import re
import json
from pathlib import Path

import pypdf
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-5.4-mini'

## Função de parsing de PDF

In [ ]:
def parse_cv(path: str) -> dict:
    """Extrai texto de um CV (PDF ou TXT) e estrutura em campos canonicos via LLM."""
    p = Path(path)
    if p.suffix.lower() == '.pdf':
        reader = pypdf.PdfReader(str(p))
        texto = '\n'.join((page.extract_text() or '') for page in reader.pages)
    else:
        texto = p.read_text(encoding='utf-8')

    prompt = f"""Extraia do curriculo abaixo um JSON com os campos:
- nome (string)
- email (string)
- formacao (lista de objetos: instituicao, curso, ano_conclusao)
- experiencia (lista de objetos: empresa, cargo, periodo, descricao)
- skills (lista de strings)
- anos_experiencia_total (int)

Responda APENAS com o JSON, sem markdown.

CURRICULO:
{texto}
"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    raw = resp.choices[0].message.content
    return json.loads(raw)

## Função de matching com vaga

In [ ]:
def match_score(cv_dict: dict, vaga_dict: dict) -> dict:
    """Pontua de 0 a 100 o match entre CV estruturado e vaga. Score hibrido: heuristica + LLM."""
    # Heuristica: match de skills
    cv_skills = set(s.lower() for s in cv_dict.get('skills', []))
    vaga_skills = set(s.lower() for s in vaga_dict.get('skills_obrigatorias', []))
    overlap = len(cv_skills & vaga_skills)
    total = max(len(vaga_skills), 1)
    score_heuristico = int(100 * overlap / total)

    # LLM: julgamento e justificativa
    prompt = f"""Voce e um recrutador experiente. Avalie o match entre o candidato e a vaga.
Responda JSON com os campos: score_llm (0-100), justificativa (string curta em pt-BR).

VAGA:
{json.dumps(vaga_dict, ensure_ascii=False, indent=2)}

CANDIDATO:
{json.dumps(cv_dict, ensure_ascii=False, indent=2)}
"""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    out = json.loads(resp.choices[0].message.content)
    score_llm = int(out.get('score_llm', 0))
    score_final = int(0.4 * score_heuristico + 0.6 * score_llm)
    return {
        'score_final': score_final,
        'score_heuristico': score_heuristico,
        'score_llm': score_llm,
        'justificativa': out.get('justificativa', ''),
    }

## TODO: viés / fairness

In [ ]:
# precisa implementar análise de viés (gender, age, etc.)
# - disparate impact ratio
# - equal opportunity difference
# - estratificar score por genero inferido e faixa etaria
# - cuidado: inferir genero/idade tambem e' uma decisao etica
# - conversar com DPO antes de qualquer coisa

## Teste com CVs de exemplo

In [ ]:
vaga_exemplo = {
    'titulo': 'AI Engineer Pleno',
    'descricao': open('data/vagas/vaga-001-ai-engineer.txt', encoding='utf-8').read(),
    'skills_obrigatorias': ['Python', 'FastAPI', 'Docker', 'LLM', 'PostgreSQL'],
}

cvs_dir = Path('data/cvs-exemplos')
for cv_path in sorted(cvs_dir.glob('*.txt')):
    print(f'\n=== {cv_path.name} ===')
    cv = parse_cv(str(cv_path))
    resultado = match_score(cv, vaga_exemplo)
    print(f"Score final: {resultado['score_final']}")
    print(f"Justificativa: {resultado['justificativa']}")

## TODO: deploy

- API (FastAPI)
- UI para recrutador (Streamlit?)
- Auth multi-tenant
- Persistencia
- Log auditavel (Art. 20 LGPD)
- Observabilidade + custo
- Pitch para Comite de Etica